# EF-02 Setup — Install Everything Once

**Run this notebook first** before any other doc or production notebook.

Each section installs one group of packages in its **own cell** so you see live `pip` output and know exactly which step finished (or failed).

---

## What you need before starting

| Requirement | Notes |
|-------------|-------|
| **Python 3.10+** | Check in Section 1 |
| **Internet** | Downloads packages and (first time) the FinBERT model weights |
| **Jupyter** | VS Code, JupyterLab, or Anaconda — any works |
| **~2–3 GB disk** | Only if installing FinBERT (Transformers + model cache) |

---

## Which sections do I need?

| If you only run… | Install sections |
|------------------|----------------|
| Scraper + consolidation + charts | 1 → 4, 8 (skip 5 FinBERT) |
| LLM classifier doc (`demo_llm_classifier`) | 1 → 4, **6**, 8–9 (skip 5 unless benchmarking FinBERT) |
| Full benchmarking doc (incl. FinBERT) | **All sections** 1 → 9 |
| Production notebooks `01_`–`06_` | **All sections** 1 → 9 |

> **Tip:** Run cells **top to bottom**. If a cell errors, fix it before continuing — later verify steps depend on earlier installs.

---
## Section 1 — Check your Python environment

No installs yet. This cell only prints your Python version and where it lives.

You want **3.10 or newer**. If this fails, select a different kernel (e.g. Anaconda base) before continuing.

In [ ]:
import sys
from pathlib import Path

print("Python version:", sys.version.split()[0])
print("Executable:    ", sys.executable)
print("Working dir:   ", Path.cwd())

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print("\n⚠ Python 3.10+ recommended. Upgrade or switch kernel before installing.")
else:
    print("\n✓ Python version looks good.")

---
## Section 2 — Core data & charts *(small, ~30 seconds)*

Used by almost every notebook: pandas tables, numpy math, matplotlib/seaborn charts, scipy stats.

**Notebooks:** `02_cleaning`, `05_consolidation`, `06_visualisation`, all `demo_*.ipynb`

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy scikit-learn

---
## Section 3 — Web scraping *(small, ~15 seconds)*

Download HTML and parse headlines.

**Notebooks:** `01_scraper`, `demo_scraper_logic`

In [ ]:
%pip install requests beautifulsoup4 lxml

---
## Section 4 — Rule-based NLP: TextBlob & VADER *(small, ~30 seconds)*

Lightweight sentiment baselines — no GPU required.

**Notebooks:** `03_benchmarking`, `demo_benchmarking` (Sections 2–3)

Section 7 below downloads the VADER lexicon and TextBlob corpora **after** this install.

In [ ]:
%pip install textblob nltk

---
## Section 5 — FinBERT via Hugging Face Transformers *(large — can take several minutes)*

⏳ **This is the biggest install.** Watch the output for download progress.

### Do we use PyTorch directly?

**No.** No notebook in this project has `import torch`. We only use:

```python
from transformers import pipeline
finbert = pipeline("text-classification", model="ProsusAI/finbert")
```

That appears in `03_benchmarking.ipynb` and `demo_benchmarking.ipynb` (Section 4). Under the hood, Hugging Face pulls in **PyTorch** as the model engine — but you never touch it yourself.

This cell installs `transformers` with its PyTorch backend (`transformers[torch]`). The **first** time you run FinBERT in a notebook, it also downloads model weights (~400 MB) — that happens later, not here.

**Skip this section** if you will not run FinBERT.

In [ ]:
%pip install "transformers[torch]"

---
## Section 6 — API clients *(small, ~15 seconds)*

Groq and OpenAI SDKs for LLM classification and benchmarking. `python-dotenv` loads keys from `../.env`.

**Notebooks:** `04_classifier`, `demo_llm_classifier`, `demo_benchmarking` (Sections 5–7)

In [ ]:
%pip install python-dotenv groq openai

---
## Section 7 — Download NLTK & TextBlob data files

Packages alone are not enough — VADER needs a lexicon file and TextBlob needs a small corpus. This cell prints each download as it completes.

In [ ]:
import nltk
from textblob import TextBlob

print("Downloading VADER lexicon...")
nltk.download("vader_lexicon", quiet=False)
print("\nDownloading TextBlob corpora...")
import subprocess
import sys
subprocess.run([sys.executable, "-m", "textblob.download_corpora"], check=False)
print("\n✓ NLP data downloads finished.")

---
## Section 8 — Verify imports

Quick smoke test. Each block prints ✓ or ✗ so you know what is missing.

If the FinBERT line shows ✗, that is fine when you skipped Section 5.

In [ ]:
def check(label, fn):
    try:
        fn()
        print(f"  ✓ {label}")
        return True
    except Exception as e:
        print(f"  ✗ {label} — {e}")
        return False

ok = []

print("Core data & charts")
ok.append(check("pandas", lambda: __import__("pandas")))
ok.append(check("numpy", lambda: __import__("numpy")))
ok.append(check("matplotlib", lambda: __import__("matplotlib.pyplot")))
ok.append(check("seaborn", lambda: __import__("seaborn")))
ok.append(check("scipy", lambda: __import__("scipy.stats")))
ok.append(check("sklearn", lambda: __import__("sklearn.metrics")))

print("\nWeb scraping")
ok.append(check("requests", lambda: __import__("requests")))
ok.append(check("beautifulsoup4", lambda: __import__("bs4")))

print("\nRule-based NLP")
ok.append(check("textblob", lambda: __import__("textblob")))
ok.append(check("nltk.vader", lambda: __import__("nltk.sentiment")))

print("\nFinBERT (optional — skip if you skipped Section 5)")
ok.append(check("transformers", lambda: __import__("transformers")))

print("\nAPI clients")
ok.append(check("dotenv", lambda: __import__("dotenv")))
ok.append(check("groq", lambda: __import__("groq")))
ok.append(check("openai", lambda: __import__("openai")))

passed = sum(ok)
total = len(ok)
print(f"\n{'='*50}")
print(f"Result: {passed}/{total} checks passed")
if passed == total:
    print("✓ Environment ready — open 00_pipeline_overview.ipynb next.")
else:
    print("⚠ Re-run the install section that failed, then run this cell again.")

---
## Section 9 — API keys (`.env` file)

LLM notebooks need keys in **`EF-02/.env`** (one folder above `docs/`).

Create the file if it does not exist:

```
GROQ_API_KEY=gsk_your_key_here
OPENAI_API_KEY=sk-your_key_here
```

| Key | Required for | Get one at |
|-----|--------------|------------|
| `GROQ_API_KEY` | Classifier + Groq Llama demos | [console.groq.com](https://console.groq.com) |
| `OPENAI_API_KEY` | GPT-4o-mini benchmarking only | [platform.openai.com](https://platform.openai.com) |

Groq alone is enough for `demo_llm_classifier` and most of production `04_classifier`.

Run the cell below to check whether keys are visible (it never prints the actual secret).

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env")
load_dotenv(env_path)

print(f".env path: {env_path.resolve()}")
print(f".env exists: {env_path.exists()}\n")

for name in ["GROQ_API_KEY", "OPENAI_API_KEY"]:
    value = os.getenv(name)
    if value:
        masked = value[:4] + "..." + value[-4:] if len(value) > 8 else "(set)"
        print(f"  ✓ {name} loaded — {masked}")
    else:
        print(f"  ✗ {name} not found — add it to .env if you need LLM notebooks")

---
## Section 10 — Where to go next

| Step | Notebook |
|------|----------|
| 1 | [`00_pipeline_overview.ipynb`](00_pipeline_overview.ipynb) — map of the whole project |
| 2 | Any `demo_*.ipynb` — progressive tutorials |
| 3 | `../notebooks/01_`–`06_` — full production pipeline |

### Full package list (reference)

If you prefer a single terminal install instead of this notebook:

```bash
pip install pandas numpy matplotlib seaborn scipy scikit-learn
pip install requests beautifulsoup4 lxml
pip install textblob nltk
pip install "transformers[torch]"   # only for FinBERT benchmarking
pip install python-dotenv groq openai
python -m textblob.download_corpora
```

Then download VADER in Python: `nltk.download("vader_lexicon")`